# Mobile Phone Price Prediction — Exploratory Data Analysis & MLOps Pipeline

This notebook covers the complete machine learning lifecycle for the **Mobile Price Classification** task:
1. **Exploratory Data Analysis (EDA)**
2. **Feature Correlation & Analysis**
3. **Data Preprocessing & Scaling**
4. **Multi-Model Training & Benchmarking** (Decision Tree, Random Forest, Logistic Regression, Gradient Boosting, SVC)
5. **Model Evaluation & Quality Gate**
6. **Artifact Export & Verification for Production Serving**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.pipeline import Pipeline

print('Libraries successfully imported!')

## 1. Load Dataset & Inspect Schema

In [ ]:
data_path = Path('../data/raw/mobile_price_train.csv')
df = pd.read_csv(data_path)
print(f'Dataset Shape: {df.shape}')
print('\nMissing values per column:')
print(df.isnull().sum())
print('\nClass distribution (price_range):')
print(df['price_range'].value_counts())
df.head()

## 2. Feature Correlation with Price Range

In [ ]:
plt.figure(figsize=(12, 8))
corr = df.corr()
sns.heatmap(corr, cmap='coolwarm', annot=False)
plt.title('Mobile Phone Features Correlation Heatmap')
plt.show()

print('\nFeatures sorted by correlation with price_range:')
print(corr['price_range'].sort_values(ascending=False))

## 3. Train-Test Split & Feature Preprocessing

In [ ]:
X = df.drop(columns=['price_range'])
y = df['price_range']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train shape: {X_train_scaled.shape}, Test shape: {X_test_scaled.shape}')

## 4. Multi-Algorithm Benchmark & Model Comparison

In [ ]:
models = {
    'Decision Tree': (DecisionTreeClassifier(max_depth=6, random_state=42), False),
    'Random Forest': (RandomForestClassifier(n_estimators=200, random_state=42), False),
    'Gradient Boosting': (GradientBoostingClassifier(n_estimators=150, random_state=42), False),
    'SVC': (SVC(probability=True, random_state=42), True),
    'Logistic Regression': (LogisticRegression(max_iter=1000, C=1.0, random_state=42), True),
}

for name, (clf, use_scaled) in models.items():
    X_tr = X_train_scaled if use_scaled else X_train
    X_te = X_test_scaled if use_scaled else X_test
    clf.fit(X_tr, y_train)
    preds = clf.predict(X_te)
    acc = accuracy_score(y_test, preds)
    print(f'{name:22}: Accuracy = {acc:.4f} ({acc*100:.2f}%)')

## 5. Best Model Evaluation & Confusion Matrix (Logistic Regression: 96.50%)

In [ ]:
best_clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
best_clf.fit(X_train_scaled, y_train)
y_pred = best_clf.predict(X_test_scaled)

print('Classification Report:')
class_names = ['Low Cost (0)', 'Medium Cost (1)', 'High Cost (2)', 'Very High Cost (3)']
print(classification_report(y_test, y_pred, target_names=class_names))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Mobile Price Classification Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 6. Single Phone Inference Verification

In [ ]:
sample = X_test.iloc[0:1]
sample_scaled = scaler.transform(sample)
pred = best_clf.predict(sample_scaled)[0]
probs = best_clf.predict_proba(sample_scaled)[0]

print(f'Actual: {y_test.iloc[0]} ({class_names[y_test.iloc[0]]})')
print(f'Predicted: {pred} ({class_names[pred]})')
print(f'Confidence: {probs[pred]*100:.2f}%')